<a href="https://colab.research.google.com/github/sadot04/Inteligencia_artificial/blob/main/Trabajo_en_grupo_DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0) Imports y configuración
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from tensorflow.keras import layers, models, callbacks

# Reproducibilidad
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

# 1) Cargar el dataset de clima

CSV_PATH = "weather_prediction_dataset.csv"
df = pd.read_csv(CSV_PATH)

print("Forma del dataset:", df.shape)
print("Columnas:", list(df.columns))

# 2) Selección de variables y objetivo (y)
# Objetivo: Regresión para predecir BASEL_temp_mean (temperatura media en Basel).

y = df["BASEL_temp_mean"].astype(float).values

# Features: Seleccionamos un subconjunto para simplicidad
X = df[[
    "DATE", "MONTH",
    "BASEL_cloud_cover", "BASEL_humidity", "BASEL_pressure", "BASEL_global_radiation", "BASEL_precipitation", "BASEL_sunshine"
]].copy()

# 3) Feature engineering liviano y limpieza
X["YEAR"] = pd.to_datetime(X["DATE"], format="%Y%m%d").dt.year
X["DAY_OF_YEAR"] = pd.to_datetime(X["DATE"], format="%Y%m%d").dt.dayofyear
X.drop("DATE", axis=1, inplace=True)  # Eliminamos DATE original

X["MONTH"] = X["MONTH"].astype(str)

# 4) Definición de columnas por tipo
num_cols = [
    "BASEL_cloud_cover", "BASEL_humidity", "BASEL_pressure", "BASEL_global_radiation", "BASEL_precipitation", "BASEL_sunshine"
]

cat_cols = ["MONTH"]

# 5) Preprocesamiento con ColumnTransformer
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

# 6) Split train/test estratificado
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Ajustar transformadores en train y transformar ambos
X_train = preprocess.fit_transform(X_train_df)
X_test = preprocess.transform(X_test_df)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Dimensiones de entrada:", X_train.shape[1])

# 7) Definir y compilar el modelo (MLP para regresión)
def build_model(input_dim: int) -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(1, activation="linear")
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
        metrics=["mae"]
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

# 8) Callbacks para entrenamiento
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=12,
                            restore_best_weights=True),
    callbacks.ModelCheckpoint("weather_best.keras", monitor="val_loss", mode="min",
                              save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6)
]

# 9) Entrenamiento
hist = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=cbs,
    verbose=1
)

# 10) Evaluación en test
y_pred = model.predict(X_test).ravel()

print("\nError Cuadrático Medio (MSE):", mean_squared_error(y_test, y_pred))
print("Error Absoluto Medio (MAE):", mean_absolute_error(y_test, y_pred))
print("Coeficiente de Determinación (R²):", r2_score(y_test, y_pred))

# 11) Inferencia robusta: función predict_one
def predict_one(sample: dict) -> float:
    """
    Recibe un diccionario 'crudo' con las llaves esperadas (e.g., MONTH, BASEL_cloud_cover, etc.).
    Calcula features temporales internamente si se proporciona DATE.
    Devuelve la temperatura media predicha para Basel.
    """
    # Convertir a DataFrame con columnas en orden esperado
    s = pd.DataFrame([sample])

    # Feature engineering consistente
    if "DATE" in s.columns:
        s["YEAR"] = pd.to_datetime(s["DATE"], format="%Y%m%d").dt.year
        s["DAY_OF_YEAR"] = pd.to_datetime(s["DATE"], format="%Y%m%d").dt.dayofyear
        s.drop("DATE", axis=1, inplace=True)
    s["MONTH"] = s["MONTH"].astype(str)

    # Aplicar exactamente el mismo procesamiento
    s_proc = preprocess.transform(s[X.columns])
    s_proc = s_proc.astype(np.float32)

    # Predecir
    pred = model.predict(s_proc).item()
    return pred

# Ejemplo de uso
sample = {
    "DATE": 20000101,
    "MONTH": 1,
    "BASEL_cloud_cover": 8,
    "BASEL_humidity": 0.89,
    "BASEL_pressure": 1.0286,
    "BASEL_global_radiation": 0.2,
    "BASEL_precipitation": 0.03,
    "BASEL_sunshine": 0.0,
}

pred = predict_one(sample)
print(f"Temperatura media predicha para Basel: {pred:.2f} °C")

TensorFlow: 2.19.0
Forma del dataset: (3654, 165)
Columnas: ['DATE', 'MONTH', 'BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation', 'BASEL_precipitation', 'BASEL_sunshine', 'BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max', 'BUDAPEST_cloud_cover', 'BUDAPEST_humidity', 'BUDAPEST_pressure', 'BUDAPEST_global_radiation', 'BUDAPEST_precipitation', 'BUDAPEST_sunshine', 'BUDAPEST_temp_mean', 'BUDAPEST_temp_max', 'DE_BILT_cloud_cover', 'DE_BILT_wind_speed', 'DE_BILT_wind_gust', 'DE_BILT_humidity', 'DE_BILT_pressure', 'DE_BILT_global_radiation', 'DE_BILT_precipitation', 'DE_BILT_sunshine', 'DE_BILT_temp_mean', 'DE_BILT_temp_min', 'DE_BILT_temp_max', 'DRESDEN_cloud_cover', 'DRESDEN_wind_speed', 'DRESDEN_wind_gust', 'DRESDEN_humidity', 'DRESDEN_global_radiation', 'DRESDEN_precipitation', 'DRESDEN_sunshine', 'DRESDEN_temp_mean', 'DRESDEN_temp_min', 'DRESDEN_temp_max', 'DUSSELDORF_cloud_cover', 'DUSSELDORF_wind_speed', 'DUSSELDORF_wind_gust', 'DUSSELDORF_humidity', 

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 32)             │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,697 (6.63 KB)

 Trainable params: 1,697 (6.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 148.7981 - mae: 10.3396 - val_loss: 94.6833 - val_mae: 8.2148 - learning_rate: 0.0010
Epoch 2/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 69.3805 - mae: 6.9017 - val_loss: 32.0023 - val_mae: 4.7460 - learning_rate: 0.0010
Epoch 3/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 33.8405 - mae: 4.7654 - val_loss: 20.1693 - val_mae: 3.6969 - learning_rate: 0.0010
Epoch 4/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 24.5805 - mae: 3.9866 - val_loss: 14.9337 - val_mae: 3.1549 - learning_rate: 0.0010
Epoch 5/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 21.1106 - mae: 3.6756 - val_loss: 12.9507 - val_mae: 2.9448 - learning_rate: 0.0010
Epoch 6/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.2096 - mae: 3.3332 - val_loss: 12.2704 - val_mae: 2.8652 - learning_rate: 0.0010
Epoch 7/200
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.8249 - mae: 3.3759 - val_loss: 11.9227 - val_mae: 2.8273 - learning_rate: 0.00